## Google Colab Setup

**GPU Required:** Before running, enable GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** (or better)
3. Click **Save**

In [1]:
# Install dependencies (skip if already done)
import os

# Set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

if not os.path.isfile("FOUNDRY_READY"):
    print("Installing rc-foundry...")

    # Uninstall torchvision first to avoid operator conflicts
    os.system("pip uninstall -y torchvision")

    # Install rc-foundry
    os.system("pip install -q 'rc-foundry[all]'")

    # Mark as ready
    os.system("touch FOUNDRY_READY")

    print("Done!")
else:
    print("rc-foundry already installed.")

rc-foundry already installed.


In [2]:
# Download model weights (skips already-downloaded models automatically)
# In total, ~6GB (3GB for RFD3, 3GB for RF3, <100MB for MPNN); may take a few minutes depending on your connection speed
os.system("foundry install rfd3 ligandmpnn rf3")

0

# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Generation** | RFD3 | Generate novel proteins via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation via Refolding** | RF3 | Predict the structure from designed sequence to validate designability |

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → RF3 (validation) → RMSD comparison
```

---

In [3]:
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

## Section 1: All-Atom Generation with RFD3

RFdiffusion3 (RFD3) generates *de novo* all-atom proteins that meet specific conditioning requirements.

**Parameters Used** *(many more are available for more complex protein design tasks)*:
- `length`: Target protein length in residues
- `diffusion_batch_size`: Number of structures to generate per batch
- `n_batches`: Number of batches to run

**Outputs:** Dictionary of `RFD3Output` objects.

In [4]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

# Set seed for reproducibility
seed_everything(0)

# Configure RFD3 inference
config = RFD3InferenceConfig(
    specification={
        'input': "/content/fold_basic_invasin_model_0.cif",
        'contig': "30-50,C113-120,30-80,/0,A1-377,/0,B1-480",
        'select_hotspots': "A70,A256,B160",
        'infer_ori_strategy': 'hotspots',
    },
    diffusion_batch_size=2,  # Generate 2 structures per batch
)

# Initialize engine and run generation
model = RFD3InferenceEngine(**config)
outputs = model.run(
    inputs=None,      # None for unconditional generation
    out_dir=None,     # None to return in memory (no file output)
    n_batches=2,      # Generate 1 batch
)

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
INFO:foundry:cuEquivariance is available and will be used.
DEBUG:transforms:Debug mode is on
INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:lightning.pytorch.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has T

In [5]:
# Inspect RFD3 outputs and extract the generated structures
#for idx, data in outputs.items():
#    print(f"Batch {idx}: {len(data)} structure(s)")
#    print(f"  Output type: {type(data[0]).__name__}")
#    print(f"  AtomArray: {data[0].atom_array}")

# Extract the first generated structure for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

# Visualize the generated structure
view(atom_array)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [6]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": None,        # Return results in memory
    "write_structures": False,
    "write_fasta": False,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 8,         # Generate 10 sequences per structure
        "remove_waters": True,
        "chains_to_design": "A"
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = []
for first_key in outputs.keys():
  atom_array = outputs[first_key][0].atom_array
  mpnn_outputs.append([model.run(input_dicts=input_configs, atom_arrays=[atom_array]),int(first_key[1:])])

---

## Section 3: Structure Prediction with RF3

RF3 (RoseTTAFold 3) predicts protein structures from sequences. By re-folding the MPNN-designed sequence, we can validate whether the design is likely to adopt the intended backbone structure.

**Outputs:** `RF3Output` objects containing:
- `atom_array`: Predicted structure as Biotite AtomArray
- `summary_confidences`: Overall confidence metrics (pLDDT, PAE, pTM, etc.)
- `confidences`: Per-atom/residue confidence scores

**Confidence Metrics:**
| Metric | Description |
|--------|-------------|
| pLDDT | Per-residue confidence (0-1, higher is better) |
| PAE | Predicted Aligned Error (lower is better) |
| pTM | Predicted TM-score |
| ranking_score | Overall model quality score |

In [7]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Initialize RF3 inference engine
inference_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)

# Create input from the MPNN-designed structure (first design)
# This re-folds the sequence to validate it adopts the intended structure
rf3_outputs = []
for j, mpnn_output_x in enumerate(mpnn_outputs):
  for i, item in enumerate(mpnn_output_x[0]):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts]
    )
    print(f"Sequence {mpnn_output_x[1]+1}_{i+1}: {seq_1letter}")
    input_structure = InferenceInput.from_atom_array(item.atom_array, example_id=f"{mpnn_output_x[1]+1}_{i+1}")
    rf3_outputs.append([inference_engine.run(inputs=input_structure),seq_1letter])

Sequence 1_1: MEVSVPGTDVFDPNARARLARDIARAVALKPPAIRLLADATVQSRVTAEMLEAIAETAPADTRVTVESDVVDPAVAASPAMAAVDLLVSSRAEAVAIARARGVAARLGGSLEVLENTPLTEVKPGDKKKLKPEDIVEISPQKLTLYLDVGVPQTFTLSYKRPVDYPIDLLILADRGFIMREALKTLRSLGTELMAELSKISSDVRIGFATFCNAPVAPFVDMTPAKRKNPCGPDEDCAPNFAFRLVLPLTSDGALFDERVAAERISAHLIYRTGMTLGIMQAAVCTEKVGWRDVTRILVVMTLHGLLSAGDGALAGIVTPNDGKCHLEDGRLTNATTYDYPSVAQLRDELTKNRIRVIFDVGASQMPLYEELASYIPNSAVRKLSGDASNVIDNILEALKELLSTVTLEHSELPAGVTIAFTSNCVNGVVGTGDAGKVCTGIAEGDTVTFDVSVTATVRPPESSTSFDIFVLGSFQKTTVNLVFRANLDTSPEKVLTVTGEAGSLFGYSLAFHHQLRPADRRLLLIGAPNAAALPNQQATRTGGVFAWDPAVAGAATRIDFGNWNDPSTEDRENSQWGYAIASQGPGGAVVTCAPNYTRITNVGTANETRYVTGRCWVLAEDLTVRYPWHGDHYEPAVGKTQGIEYYGRAQVGVAVTFSPDGRYLILGAPGAYNFKGIVIVLPIPEKFAELGITDDGPFAVGGWDDDDISRVPLSAGAAAAEEEEEEEEDEEPGEEEEEEPPPPPRPPRPPGGTNESFLGFSLDCAKGIVSKDELTIVAGAPGHKARGAVFLLAINYDTKELVVVHRFEGPYLFSQFGYSVAVVDLSANGWKDIVIGAPQATTKDGKVGGAVYVYKNNNGNWENVTPTVLNGPANSQFGIAVKNVGDIDNNGYDDIAVGAPLDGLGKVYIYKGSSSGLNTTPSQVLEGTVPYFGYAIAGNLDLFGNGLPDVAVGSLSDTVTVYRS


INFO:rf3.inference_engines.rf3:[rank: 0] Loading checkpoint from /root/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: 1_1


Sequence 1_2: EAISVPGVDANDPNAAARLARQIRRAVALKPPAIVLRAHATVLSRVTAEMLATIAATAPKDTKVTVVSDVVAPEVAASPHMAAIDLLISSDARAVEIARARGVKAVLGGSLEVLKNTPVTVVKPGDKKKLKPEDIVSISPQELTLTLDVGVPVTFTLTYKKPYDYPIDILFLADRGAIMRAALETLRSLGTELMAELSKLTSDVRLGYATFCNRPVAPFVDTTPKKRKNPCGPDEDCAPNYAFRLVLPLTSDGALFDERVGAERISAHLILRSGITLGIMQAAVCTDKVGWRNRRNILVVMTLTGLLSRGDGALAGIVTPNDGKCHLEDGLLTNATTYDFPSVAQLVELLTKNRIQVIFAVGASQLPMYKELASLIPKSAVVKLAGDASNIIQNILDALATLDSTVTLKHSELPAGVTIAFTSHCVNGVVGTGAAGRVCTGIAVGDTVTFDVSVTATVLPPEASTSFKIFVLGSFESTTVNLVFAANLDTTPSKVITVTGEAGSLFGYSLAFHHQTSPADRRLLLIGAPNAAALPNQQATRTGALYAYDPAKTGAATRIDVGNWNDPTTEDRENSQWGYAVASQGPGGRVVTCAPNYTRVTNVGTEDETRYRTGRCWVLAEDLTVRYPWDGDGYAPAVGKTQGIHAYGLAQVGVAVTFSPDGRYLILGAPGAYNFRGIVIVLPIPEVFEKEGITKDGPFAVGGWDDDDISLVPLDFGSSASKKKKKKKKKKKKGKKKKKKPKPPKRPPRPPGGTNESFLGFSLDCAKGIVSKDELTIVAGAPGHHSRGAVFLLKINHETNRLVVVHRFEGPALFSQFGYSVAVVDLNQNGWKDIVIGAPQYYTTDGKVGGAVYVYKNNNGKWENVTPTVLNGPANSQFGIAVKNIGDIDLNGYDDIAVGAPLDGLGRVYIYAGSSNGLNTTPSQVLEGTVPYFGYAIAGNMDLTGNGRPDVAVGSLSDTVTVYTA


INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: 1_2


Sequence 1_3: RAESVPGTDVFDPNAAADLARLIRRAVARRPPAIRLLAHATVQSRVTAEMLAAVAAEAPADTRVEVVDDVVAPEVAASPEMAAVDLLVSSRPDAVAIARARGVAARLGGSLEVLENTPLTVVEPGKKKKLKPEDIVEISPQKLKLTLDVGVPVTFTLSYKRPYDYPIDILFLADRGFIMKEALKTLKKLGTELMKELSKISSDVRLGFATFCNVPAAPFVDTTPAKRKNPCGPDENCAPNYAFRLVLPLTSDGKLFDERVAKQKISAHLILRSGITLGIMQAAVCTEKVGWRKTRRILVVMTMTGLLSRGDGALAGIVLPNDGKCHLVDGMLTNATTYDFPSVAQLRDILTENKISVIFAVTARQMPMYEELSSYIPNSAVVKLLGDASNIIENILEALKKLDSTVTLEVSELPEGVTISFTSYCVNGVVGTGAAGKVCTGINVGDTVKFAVTVTATVRPPEATTSFDIFVLGSFQKTTVILEFAANLDTSPEKVKTVTGAAGSLFGYSLAFHHQTSPADRRLLLIGAPNDAAKPNQKATKTGAVYAWDPAVEGAATRIDFGNWNDPTTEDWENSQWGYAIASQGPGGRVVTCAPNYTYITNVGTADEQRYVTGRCCVLAEDLTERYPYDGCHYDPARGKTQGIAYYGRAQVGVAVTFSPDGEFLILGAPGAFNFRGIVIVLPLQPVFDKKGIKKFGPFAVGGWDDDDISLVPADPGAAELEEEEEEEEEEEPEEEEEEEPPPPPRPPRPPGLTNESYLGFSLDCAKGIVSKDKLTIVAGAPGAHSRGAVVLLAIDEKTNRLDVVHTFYGESLYSRFGYSVAVVDLNNNGWKDIVIGAPEYTTTDGKVGGAVYVYKNNNGNWSNVTPTVLNGPANSLFGIAVKNVGDIDLNGYDDIAIGAPQDGLGKVYIYKGSSSGLNTTPSQVLEGTVPYFGYAIAGNMDLTGNGKPDVAVGSLSDTVTIYTS


...

INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: 2_6


Sequence 2_7: ATLAQLLREEFGPSPDRVTVTVTITIPPELSAIITSIGLGGHGSVDHAIEKVVDVPKGATKLTVTVTMLDPGSSFEAYLARGDEVIARVSAAPLRGPPSAPTVDIDGARARPLRVRGSLEVLKNTPVTVVKPGEKKKYKPEDIVEISPQELRLTLDVGVPQTFTLTYKRPYNYPIDILFLQDLSYSMEKALKTLKSLGTKLMEELSKRTDDVRLGFAAFVDAPVAPFVDTTPQKRKNPCNSSVDCRPNFAFRLVLPLTNDGKLFDERVGKQKISNHITYRTGMTLGIMQAAVCTEKVGWRNTINLLVVATDHGLHSLGDGALAGIVTPNDGKCHLENGLLTNATTYDYPSIAQLRDELTKNRIRVIFAVTERQMPLYEELASYIPNSAVRTLAPDASNIIDNILEAMDELDSKVTLEHSALPEGVTISFTSHCVNGVVGTGDAGKVCTNIKVGDTVTFDVTVTATVRHPQAEFSFDIFVLGSFQKTTVILRFRANLDTTPANVITVTGAAGSLFGYSLAFHHQLEPEDRRLLLIGAPNDAARPNQKATKTGAVYAWDPAQAGAATRIDFGNWNDPTTEDKENSQWGYSIASQGPGGRVVTCAPNYTYITNVGTSNETRYITGRCTVLAEDLTVRYPYDGDHYDPARGKTQGAAYYGRAQVGVAVTFSPDGEFLILGAPGAFLNTGIVIVLPIPEVFEKKGIKKFGPFAVGGWDDEDASLVPARGGAAEEEEEEEPEEEPDEEEEEEEEPEPPPEPPRPPGGTTESFLGFSLDCAKGIVSKDKLTIVAGAPGHRARGAVFLLAIDEETNRLVVVHVFEGPYLFSKFGYSVAVVDLSNNGWKDIVIGAPQAYTTDGTVGGAVYVYYNNNGKWENVKPTVLNGPANSLFGIAVKNVGDIDNNGYDDIAVGAPLDGLGKVYIYKGSANGLNTTPSQVLEGTVPYFGYSIAGNMDLTGNGLPDVAVGSLSDTVTVYTS


INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: 2_7


Sequence 2_8: ATLAQILADEFGPSPDVRTVTVTVDIPPKYSAIITSIALGGHGAVGHAFEVVVDVPKGATRLTVTVTMVDPGSSFEAALCRGDEVIALVSAAPLRGPPEAPYVYIDTSKARPRRVRGSLTVLENKPITEVKPGDKKKLKPEDIVEISPQELTLTLDVGVPQTFTLTYKKPYNYPIDILFLQDLSASMRRALETLRSLGTELMKELSKITDDVRLGYATFCNAPVEPFVDTTPQKRKNPLSPSVNAAPNFAFRLVLPLTSDGEEFNRVVGQERISGHITYRTGITRGIMQAAVCTAQVGWRNRRRILVVSTDHGIYSRGDGALAGIVLPNDGQCHLTDGLLTNATTLDYPSIAQLVEVLTKNRIRVIFDVTEDQMPMYTELSKLIPNSAVRTLAPDASNIIENILKALEELDSTVTLEHSELPEGVSISFTSHCVNGVTGTGEAGKVCTGIKVGDTVTFDVSVTASVRHPQATFSFDIRVLGSFQKTKVNLVFAANLDTRPENVITVTGAAGSLFGYSLAFHHQTSPEDRRLLLIGAPNAAARPNQKATRTGALFAYDPAVTGAATEINFGDWNDPTTEDRENSMWGHAVASQGPGGRVVTCAPNYTRITNVGTADETRNVTGRCTVLSEDLTVKNPYDGDHYDPAVGRTQGPAYWGRAQVGVAVTFSPDGRYLILGAPGAYLFTGVVIVLPIQPVFDKEGIKKDGPWAVGGWEDHDASLVPARGRASLEKEEEEEEEEEEEEEEEEEEPEPPEEPPRPLGLTNESLLGFSLDCAKGIVSKDELTIVAGAPRAFSRGAVVLLAIDRETNRLVPVHVFFGPSLFSQFGYSVAVVDLSANGWKDIVIGAPQYYTKDGTVGGAVFVYKNNNGKWENVTPTILYGPKNSQFGIAVKNVGDIDNNGYDDIAIGAPLDGLGKVYIYSGSSSGLNTTPSQVLEGTVPYFGYAIAGNMDLFGNGLPDVAVGSLSDTVTIYRA


INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: 2_8


In [26]:
import pandas as pd

# Prepare a list to collect data for the DataFrame
df_rows = []

# Loop through each RF3 output and collect the desired information
for sublist in rf3_outputs:
  rf3_output = sublist[0]
  row_data = rf3_output[list(rf3_output.keys())[0]][0].summary_confidences.copy() # Start with confidence metrics
  row_data['atom_array'] = rf3_output[list(rf3_output.keys())[0]][0].atom_array # Add the atom_array object
  row_data['name'] = list(rf3_output.keys())[0]
  # Assuming mpnn_outputs is structured as [[seq1, seq2, ...]]
  # and rf3_outputs elements correspond to mpnn_outputs[0][k]
  i,j = list(rf3_output.keys())[0].split("_")
  row_data['rfd3_num'] = int(i)
  row_data['mpnn_num'] = int(j)
  row_data['sequence'] = sublist[1]
  row_data['example_id'] = list(rf3_output.keys())[0] # Add the example_id for easy reference

  df_rows.append(row_data)

# Create the DataFrame from the collected rows
rf3_df = pd.DataFrame(df_rows)

# Display the first few rows of the DataFrame and its columns for verification
print("RF3 DataFrame created successfully!")
print(rf3_df.head())
print("\nDataFrame columns:", rf3_df.columns.tolist())

RF3 DataFrame created successfully!
            chain_ptm                                 chain_pair_pae_min  \
0  [0.71, 0.69, 0.75]  [[None, 25.35, 24.66], [None, None, 18.76], [N...   
1  [0.73, 0.69, 0.74]  [[None, 25.41, 25.16], [None, None, 11.09], [N...   
2  [0.69, 0.69, 0.74]  [[None, 24.02, 20.81], [None, None, 22.92], [N...   
3   [0.72, 0.7, 0.75]  [[None, 24.97, 17.86], [None, None, 21.37], [N...   
4  [0.74, 0.69, 0.74]  [[None, 24.75, 22.69], [None, None, 21.47], [N...   

                                  chain_pair_pde_min  \
0  [[None, 12.63, 13.18], [None, None, 8.96], [No...   
1  [[None, 12.9, 12.7], [None, None, 6.44], [None...   
2  [[None, 12.04, 11.25], [None, None, 9.98], [No...   
3  [[None, 11.56, 10.26], [None, None, 9.01], [No...   
4  [[None, 12.67, 11.72], [None, None, 9.15], [No...   

                                      chain_pair_pae  \
0  [[None, 28.6, 27.88], [None, None, 26.86], [No...   
1  [[None, 28.2, 27.86], [None, None, 26.19], [No...   
2 

In [27]:
# Extract the top-ranked prediction
rf3_output = rf3_df.loc[rf3_df['overall_plddt'].idxmax()]

# Visualize the predicted structure
view(rf3_output["atom_array"])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Section 4: Validation and Export

The final step compares the RF3-predicted structure against the original RFD3-generated backbone. A low backbone RMSD indicates the designed sequence is likely to fold into the intended structure (high designability).

In [32]:
rf3_df

,chain_ptm,chain_pair_pae_min,chain_pair_pde_min,chain_pair_pae,chain_pair_pde,overall_plddt,overall_pde,overall_pae,ptm,iptm,has_clash,ranking_score,atom_array,name,rfd3_num,mpnn_num,sequence,example_id,rmsd
0,"[0.71, 0.69, 0.75]","[[None, 25.35, 24.66], [None, None, 18.76], [N...","[[None, 12.63, 13.18], [None, None, 8.96], [No...","[[None, 28.6, 27.88], [None, None, 26.86], [No...","[[None, 16.54, 17.1], [None, None, 13.93], [No...",0.7183,11.9842,23.0820,0.507059,0.228452,True,-99.7158,[ A 1 MET N N 40.011 -3...,1_1,1,1,MEVSVPGTDVFDPNARARLARDIARAVALKPPAIRLLADATVQSRV...,1_1,31.945219
1,"[0.73, 0.69, 0.74]","[[None, 25.41, 25.16], [None, None, 11.09], [N...","[[None, 12.9, 12.7], [None, None, 6.44], [None...","[[None, 28.2, 27.86], [None, None, 26.19], [No...","[[None, 16.71, 16.69], [None, None, 12.62], [N...",0.7210,11.0771,22.5425,0.524721,0.251120,True,-99.6942,[ A 1 GLU N N -6.264 -...,1_2,1,2,EAISVPGVDANDPNAAARLARQIRRAVALKPPAIVLRAHATVLSRV...,1_2,22.232927
2,"[0.69, 0.69, 0.74]","[[None, 24.02, 20.81], [None, None, 22.92], [N...","[[None, 12.04, 11.25], [None, None, 9.98], [No...","[[None, 27.97, 26.77], [None, None, 27.43], [N...","[[None, 16.31, 15.28], [None, None, 14.68], [N...",0.7171,12.0207,23.0383,0.509315,0.221089,False,0.2787,[ A 1 ARG N N 58.473 2...,1_3,1,3,RAESVPGTDVFDPNAAADLARLIRRAVARRPPAIRLLAHATVQSRV...,1_3,25.159033
3,"[0.72, 0.7, 0.75]","[[None, 24.97, 17.86], [None, None, 21.37], [N...","[[None, 11.56, 10.26], [None, None, 9.01], [No...","[[None, 28.17, 26.04], [None, None, 27.16], [N...","[[None, 16.03, 14.32], [None, None, 14.22], [N...",0.7274,11.6765,22.7784,0.519737,0.236172,False,0.2929,[ A 1 ARG N N -23.386 -1...,1_4,1,4,RAVSVPGSDVFDPRRAEDLARLLRRAVARRPPAILLDAAATVTSRV...,1_4,24.340178
4,"[0.74, 0.69, 0.74]","[[None, 24.75, 22.69], [None, None, 21.47], [N...","[[None, 12.67, 11.72], [None, None, 9.15], [No...","[[None, 28.24, 26.96], [None, None, 27.02], [N...","[[None, 16.45, 15.33], [None, None, 13.58], [N...",0.7217,11.5577,23.0604,0.503492,0.226179,False,0.2816,[ A 1 GLU N N -21.625 ...,1_5,1,5,EAVSIPGTDEFDPDRRARLARLIRRAVARRPPALVLRMDATVRSRV...,1_5,23.425186
5,"[0.7, 0.69, 0.75]","[[None, 23.54, 22.56], [None, None, 12.69], [N...","[[None, 11.02, 10.98], [None, None, 7.74], [No...","[[None, 27.56, 26.98], [None, None, 26.04], [N...","[[None, 15.4, 14.78], [None, None, 12.79], [No...",0.7212,10.8996,22.2440,0.536016,0.261270,True,-99.6838,[ A 1 ARG N N 6.125 2...,1_6,1,6,RAISVPGTDVFDPRAAERLAEQIRRAVAKKPPAITVLAHASDTSRT...,1_6,24.943628
6,"[0.73, 0.69, 0.75]","[[None, 24.8, 24.38], [None, None, 21.98], [No...","[[None, 12.28, 12.4], [None, None, 10.01], [No...","[[None, 28.15, 27.69], [None, None, 27.36], [N...","[[None, 16.51, 16.67], [None, None, 14.7], [No...",0.7205,12.4895,23.5190,0.494317,0.215865,False,0.2716,[ A 1 THR N N 12.668 -1...,1_7,1,7,TAESVPGTDVFDPEARARLARLLRRAVARNPPAITLRMHATCTSRV...,1_7,21.499285
7,"[0.74, 0.7, 0.74]","[[None, 24.03, 24.78], [None, None, 21.6], [No...","[[None, 13.16, 13.36], [None, None, 10.09], [N...","[[None, 28.22, 27.75], [None, None, 27.27], [N...","[[None, 17.24, 16.63], [None, None, 14.73], [N...",0.7246,12.2796,23.1615,0.493926,0.211965,True,-99.7316,[ A 1 ARG N N 1.030 1...,1_8,1,8,RAISVPGGDVLDPERAANLARLIRRAVAKKPPAITLLMDASVQSRV...,1_8,21.501850
8,"[0.69, 0.7, 0.73]","[[None, 24.88, 23.89], [None, None, 23.54], [N...","[[None, 12.36, 12.18], [None, None, 10.49], [N...","[[None, 28.28, 27.79], [None, None, 27.72], [N...","[[None, 17.21, 16.14], [None, None, 15.62], [N...",0.7126,12.9572,24.1132,0.449580,0.200203,True,-99.7499,[ A 1 ALA N N 4.124 -...,2_1,2,1,ATLAQILAPEKGPSIDVVTLTVHITIPPELSAVITSIALGGHGADF...,2_1,20.874166
9,"[0.71, 0.69, 0.74]","[[None, 23.56, 14.34], [None, None, 21.71], [N...","[[None, 10.12, 9.04], [None, None, 9.15], [Non...","[[None, 27.34, 25.61], [None, None, 27.05], [N...","[[None, 14.7, 13.05], [None, None, 14.04], [No...",0.7178,11.4162,22.8830,0.515258,0.244269,False,0.2985,[ A 1 THR N N -19.233 1...,2_2,2,2,TTLAQILAPELGPSSDVVTVRV

In [31]:
from biotite.structure import rmsd, superimpose
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
import numpy as np
from atomworks.io.utils.io_utils import to_cif_file

for index, row in rf3_df.iterrows():
  # Get structures for comparison
  aa_generated = outputs["_"+str(int(row['rfd3_num'])-1)][0].atom_array              # Original RFD3 backbone (from Section 1)
  aa_refolded = row["atom_array"]    # RF3-predicted structure

  # Filter to backbone atoms (N, CA, C, O)
  bb_generated = aa_generated[np.isin(aa_generated.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]
  bb_refolded = aa_refolded[np.isin(aa_refolded.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]

  # Superimpose structures and calculate RMSD
  bb_refolded_fitted, _ = superimpose(bb_generated, bb_refolded)
  rmsd_value = rmsd(bb_generated, bb_refolded_fitted)
  rf3_df.loc[index,"rmsd"] = rmsd_value

  print(f"\nBackbone RMSD: {rmsd_value:.2f} A")
  print(f"Interpretation: {'Excellent' if rmsd_value < 1.0 else 'Good' if rmsd_value < 2.0 else 'Moderate'} designability")
  if rmsd_value < 2.0:
    to_cif_file(aa_refolded,row['example_id']+"_refolded.cif")

# export rf3_df to csv
rf3_df.to_csv("all_designs.csv")


Backbone RMSD: 31.95 A
Interpretation: Moderate designability

Backbone RMSD: 22.23 A
Interpretation: Moderate designability

Backbone RMSD: 25.16 A
Interpretation: Moderate designability

Backbone RMSD: 24.34 A
Interpretation: Moderate designability

Backbone RMSD: 23.43 A
Interpretation: Moderate designability

Backbone RMSD: 24.94 A
Interpretation: Moderate designability

Backbone RMSD: 21.50 A
Interpretation: Moderate designability

Backbone RMSD: 21.50 A
Interpretation: Moderate designability

Backbone RMSD: 20.87 A
Interpretation: Moderate designability

Backbone RMSD: 27.81 A
Interpretation: Moderate designability

Backbone RMSD: 28.01 A
Interpretation: Moderate designability

Backbone RMSD: 25.59 A
Interpretation: Moderate designability

Backbone RMSD: 19.97 A
Interpretation: Moderate designability

Backbone RMSD: 18.97 A
Interpretation: Moderate designability

Backbone RMSD: 25.95 A
Interpretation: Moderate designability

Backbone RMSD: 26.72 A
Interpretation: Moderate design